# Банковские счета

Файлы `transactions_merge.csv` и `transactions_merge.json` нужны для проверки повторной загрузки

## Проверка входных данных

Деньги хранятся как `Decimal`, чтобы точно считать копейки

In [1]:
from __future__ import annotations

import csv
import json
import re
from datetime import datetime
from decimal import Decimal, InvalidOperation
from pathlib import Path


def money(value):
    if isinstance(value, bool):
        raise ValueError("Сумма должна быть числом")
    try:
        result = Decimal(str(value))
    except (InvalidOperation, TypeError, ValueError) as exc:
        raise ValueError("Сумма должна быть числом") from exc
    if (not result.is_finite() or result < 0
            or result != result.quantize(Decimal("0.01"))):
        raise ValueError("Сумма должна быть неотрицательной с точностью до копейки")
    return result


def positive_money(value):
    result = money(value)
    if result == 0:
        raise ValueError("Сумма должна быть положительной")
    return result


def parse_date(value):
    if not isinstance(value, str):
        raise ValueError("Отсутствует дата")
    for pattern in ("%Y-%m-%d %H:%M:%S", "%d/%m/%Y %H:%M"):
        try:
            return datetime.strptime(value.strip(), pattern)
        except ValueError:
            pass
    raise ValueError(f"Неверная дата: {value!r}")

## Счета и операции

In [2]:
class Account:
    _account_counter = 1000
    account_type = None

    def __init__(self, account_holder, balance=0):
        if not isinstance(account_holder, str) or not re.fullmatch(
            r"[A-ZА-ЯЁ][a-zа-яё]+ [A-ZА-ЯЁ][a-zа-яё]+", account_holder
        ):
            raise ValueError("Владелец должен быть записан как «Имя Фамилия»")
        initial = money(balance)
        Account._account_counter += 1
        self.holder = account_holder
        self.account_number = f"ACC-{Account._account_counter}"
        self._balance = initial
        self._opening_balance = initial
        self.operations_history = []
        self._events = {}  # записи по идентификаторам
        self._imported = False
        self._live_counter = 0
        self._pending_rows = []
        self._last_report = {}

    def _record(self, operation, amount, status, at=None):
        at = at or datetime.now()
        entry = {
            "date": at.isoformat(sep=" ", timespec="seconds"),
            "operation": operation,
            "amount": float(amount),
            "balance_after": float(self._balance),
            "status": status,
        }
        if self._imported:
            self._live_counter += 1
            event_id = f"LIVE-{self.account_number}-{self._live_counter}"
            entry["transaction_id"] = event_id
            self._events[event_id] = {
                "id": event_id, "date": at, "operation": operation,
                "amount": amount, "source_balance": self._balance,
                "status": status,
                "order": max(x["order"] for x in self._events.values()) + 1,
            }
        self.operations_history.append(entry)
        return entry.copy()

    def deposit(self, amount):
        amount = positive_money(amount)
        self._balance += amount
        return self._record("deposit", amount, "success")

    def withdraw(self, amount):
        amount = positive_money(amount)
        if amount > self._balance:
            return self._record("withdraw", amount, "fail")
        self._balance -= amount
        return self._record("withdraw", amount, "success")

    def get_balance(self):
        return float(self._balance)

    def get_history(self):
        return [item.copy() for item in self.operations_history]

    @staticmethod
    def _check_n(n):
        if isinstance(n, bool) or not isinstance(n, int) or n < 0:
            raise ValueError("n должно быть неотрицательным целым")

    def largest_operations(self, n=5):
        self._check_n(n)
        ordered = sorted(
            self.operations_history,
            key=lambda row: (row["amount"], row["date"]),
            reverse=True,
        )
        return [row.copy() for row in ordered[:n]]

    def latest_operations(self, n=5):
        self._check_n(n)
        ordered = sorted(
            self.operations_history, key=lambda row: row["date"], reverse=True
        )
        return [row.copy() for row in ordered[:n]]

    def load_history(self, path):
        return _load_history(self, path)

    def clean_history(self):
        return _clean_history(self)

In [3]:
class CheckingAccount(Account):
    account_type = "checking"
    allowed_operations = {"deposit", "withdraw"}


class SavingsAccount(Account):
    account_type = "savings"
    allowed_operations = {"deposit", "withdraw", "interest"}

    def withdraw(self, amount):
        amount = positive_money(amount)
        if amount > self._balance / 2:
            return self._record("withdraw", amount, "fail")
        return super().withdraw(amount)

    def apply_interest(self, rate):
        if isinstance(rate, bool):
            raise ValueError("Ставка должна быть числом")
        try:
            rate = Decimal(str(rate))
        except (InvalidOperation, TypeError, ValueError) as exc:
            raise ValueError("Ставка должна быть числом") from exc
        if not rate.is_finite() or rate <= 0:
            raise ValueError("Ставка должна быть положительной")
        amount = (self._balance * rate / 100).quantize(Decimal("0.01"))
        if amount == 0:
            raise ValueError("Начисленный процент меньше копейки")
        self._balance += amount
        return self._record("interest", amount, "success")

## Загрузка истории

При загрузке строки с опечатками, неверной датой, суммой, типом операции или счета пропускаются. Остаток пересчитывается по принятым операциям, исходный остаток нужен только до первой операции. Ограничение 50% применяется к новым снятиям, а не к уже записанной истории

In [4]:
def _read_rows(path):
    path = Path(path)
    if path.suffix.lower() == ".csv":
        with path.open(encoding="utf-8-sig", newline="") as stream:
            return list(csv.DictReader(stream))
    if path.suffix.lower() == ".json":
        with path.open(encoding="utf-8-sig") as stream:
            rows = json.load(stream)
        if isinstance(rows, list) and all(isinstance(row, dict) for row in rows):
            return rows
        raise ValueError("JSON должен содержать список объектов")
    raise ValueError("Поддерживаются только CSV и JSON")


def _normalise_row(raw, allowed):
    operation = str(raw.get("operation") or "").strip().lower()
    status = str(raw.get("status") or "").strip().lower()
    if operation not in allowed or status not in {"success", "fail"}:
        raise ValueError("Недопустимая операция или статус")
    return {
        "date": parse_date(raw.get("date")),
        "operation": operation,
        "amount": positive_money(raw.get("amount")),
        "source_balance": money(raw.get("balance_after")),
        "status": status,
    }

### Повторная загрузка

В исходных файлах нет `transaction_id`, поэтому каждой строке присваивается номер по ее месту среди строк счета. В дополнительных файлах ID указан явно. Один и тот же ID означает повтор или исправление записи, разные ID означают разные операции. Для исходных файлов это работает, пока порядок строк не меняется


In [5]:
def _merge_event(events, event, report):
    old = events.get(event["id"])
    fields = ("date", "operation", "amount", "source_balance", "status")
    if old is not None:
        if all(old[field] == event[field] for field in fields):
            report["duplicates"] += 1
        else:
            event["order"] = old["order"]
            events[event["id"]] = event
            report["updated"] += 1
        return

    event["order"] = max((item["order"] for item in events.values()), default=-1) + 1
    events[event["id"]] = event
    report["added"] += 1


In [6]:
def _replay(account, events):
    ordered = sorted(events.values(), key=lambda row: (row["date"], row["order"]))
    if not ordered:
        return account._opening_balance, account._opening_balance, []

    opening = account._opening_balance
    if not account._imported and not account.operations_history:
        first = ordered[0]
        direction = -1 if first["operation"] == "withdraw" else 1
        opening = first["source_balance"]
        if first["status"] == "success":
            opening -= direction * first["amount"]
        if opening < 0:
            raise ValueError("Невозможно восстановить начальный баланс")

    balance = opening
    history = []
    for event in ordered:
        if event["status"] == "success":
            direction = -1 if event["operation"] == "withdraw" else 1
            balance += direction * event["amount"]
            if balance < 0:
                raise ValueError("История приводит к отрицательному балансу")
        history.append({
            "transaction_id": event["id"],
            "date": event["date"].isoformat(sep=" ", timespec="seconds"),
            "operation": event["operation"],
            "amount": float(event["amount"]),
            "balance_after": float(balance),
            "status": event["status"],
        })
    return opening, balance, history

In [7]:
def _clean_history(account):
    if not account._pending_rows:
        if account._events:
            opening, balance, history = _replay(account, account._events)
            account._opening_balance = opening
            account._balance = balance
            account.operations_history = history
        return account.get_history()

    events = {key: value.copy() for key, value in account._events.items()}
    report = dict.fromkeys(("matched", "added", "updated", "duplicates", "rejected"), 0)
    position = 0
    for raw in account._pending_rows:
        if raw.get("account_number") != account.account_number:
            continue
        report["matched"] += 1
        position += 1
        if raw.get("account_type") != account.account_type:
            report["rejected"] += 1
            continue
        try:
            event = _normalise_row(raw, account.allowed_operations)
        except ValueError:
            report["rejected"] += 1
            continue
        fallback_id = f"LEGACY-{account.account_number}-{position}"
        event["id"] = str(raw.get("transaction_id") or fallback_id)
        _merge_event(events, event, report)

    if report["added"] or report["updated"]:
        opening, balance, history = _replay(account, events)
        account._events = events
        account._opening_balance = opening
        account._balance = balance
        account.operations_history = history
        account._imported = True
    account._pending_rows = []
    account._last_report = report
    return account.get_history()


def _load_history(account, path):
    account._pending_rows = _read_rows(path)
    try:
        account.clean_history()
    except Exception:
        account._pending_rows = []
        raise
    return account._last_report.copy()

## Проверка операций

In [8]:
checking = CheckingAccount("Иван Иванов", 1000)
assert checking.account_number.startswith("ACC-")
assert checking.deposit(250)["status"] == "success"
assert checking.withdraw(2000)["status"] == "fail"
assert checking.get_balance() == 1250
assert len(checking.get_history()) == 2

savings = SavingsAccount("Anna Petrova", 1000)
assert savings.withdraw(501)["status"] == "fail"
assert savings.withdraw(500)["status"] == "success"
assert savings.apply_interest(7)["amount"] == 35
assert savings.get_balance() == 535
for invalid in (-1, 0, float("nan")):
    try:
        savings.deposit(invalid)
    except ValueError:
        pass
    else:
        raise AssertionError(f"Не отклонена сумма {invalid!r}")
try:
    CheckingAccount("ivan ivanov")
except ValueError:
    pass
else:
    raise AssertionError("Не отклонено неверное имя")
print("Базовые операции и ограничения: OK")

Базовые операции и ограничения: OK


## Исходные файлы

Оба файла содержат одни и те же операции четырёх счетов. Сначала загружаем CSV, затем JSON. Номера счетов из файла задаем созданным объектам перед загрузкой

In [9]:
accounts = {}
for number, cls, holder in (
    ("ACC-100001", CheckingAccount, "Иван Иванов"),
    ("ACC-100002", SavingsAccount, "Анна Петрова"),
    ("ACC-100003", CheckingAccount, "Петр Сидоров"),
    ("ACC-100004", SavingsAccount, "Maria Smirnova"),
):
    account = cls(holder)
    account.account_number = number
    accounts[number] = account

for number, account in accounts.items():
    report = account.load_history("transactions_dirty.csv")
    before = account.get_balance()
    repeated = account.load_history("transactions_dirty.json")
    assert account.get_balance() == before
    assert repeated["added"] == repeated["updated"] == 0
    assert account.clean_history() == account.get_history()
    print(
        number, f"записей: {len(account.get_history())}",
        f"пропущено: {report['rejected']}",
        f"повторов из JSON: {repeated['duplicates']}",
        f"баланс: {account.get_balance():.2f}",
    )
ids_1 = {row["transaction_id"] for row in accounts["ACC-100001"].get_history()}
ids_4 = {row["transaction_id"] for row in accounts["ACC-100004"].get_history()}
assert "LEGACY-ACC-100001-16" not in ids_1  # diposit
assert "LEGACY-ACC-100004-1" not in ids_4   # diposit
assert all(
    row["operation"] != "interest"
    for row in accounts["ACC-100001"].get_history()
)


ACC-100001 записей: 18 пропущено: 8 повторов из JSON: 18 баланс: 4471.00
ACC-100002 записей: 19 пропущено: 8 повторов из JSON: 19 баланс: 4629.24
ACC-100003 записей: 13 пропущено: 8 повторов из JSON: 13 баланс: 5446.00
ACC-100004 записей: 20 пропущено: 11 повторов из JSON: 20 баланс: 10586.25


## Дополнительные файлы

Проверяем исправление записи, повтор, ошибочную строку и добавление операций задним числом. Два пополнения с одинаковыми датой и суммой должны сохраниться отдельно. Повторная загрузка файла не меняет баланс


In [10]:
before = {number: (len(account.get_history()), account.get_balance())
          for number, account in accounts.items()}
for file_name in ("transactions_merge.csv", "transactions_merge.json"):
    print("\n", file_name)
    for number, account in accounts.items():
        first = account.load_history(file_name)
        state = (len(account.get_history()), account.get_balance())
        repeat = account.load_history(file_name)
        assert (len(account.get_history()), account.get_balance()) == state
        assert repeat["added"] == repeat["updated"] == 0
        if file_name.endswith(".csv") and number == "ACC-100001":
            assert first["added"] == 2 and first["updated"] == 1
            assert first["rejected"] == 1
        if first["matched"]:
            print(
                number, f"новых: {first['added']}",
                f"исправлено: {first['updated']}",
                f"повторов: {first['duplicates']}",
                f"отклонено: {first['rejected']}",
                f"баланс: {state[1]:.2f}",
            )

assert accounts["ACC-100001"].get_balance() != before["ACC-100001"][1]
assert accounts["ACC-100002"].get_balance() != before["ACC-100002"][1]
assert accounts["ACC-100003"].get_balance() == before["ACC-100003"][1]
assert accounts["ACC-100001"].get_balance() == before["ACC-100001"][1] + 201
assert accounts["ACC-100002"].get_balance() == before["ACC-100002"][1] + 837
ids = {row["transaction_id"] for row in accounts["ACC-100001"].get_history()}
assert {"EXTRA-100001-1", "EXTRA-100001-2"} <= ids
live_account = accounts["ACC-100001"]
live_account.deposit(25)
state = (len(live_account.get_history()), live_account.get_balance())
live_account.clean_history()
assert (len(live_account.get_history()), live_account.get_balance()) == state
print("\nКрупнейшие операции:")
for row in accounts["ACC-100001"].largest_operations(3):
    print(row["date"], row["operation"], row["amount"])
print("Последние операции:")
for row in accounts["ACC-100001"].latest_operations(3):
    print(row["date"], row["operation"], row["amount"])


 transactions_merge.csv
ACC-100001 новых: 2 исправлено: 1 повторов: 1 отклонено: 1 баланс: 4672.00
ACC-100002 новых: 1 исправлено: 0 повторов: 0 отклонено: 0 баланс: 5406.24

 transactions_merge.json
ACC-100002 новых: 2 исправлено: 0 повторов: 0 отклонено: 0 баланс: 5466.24
ACC-100004 новых: 0 исправлено: 0 повторов: 0 отклонено: 1 баланс: 10586.25

Крупнейшие операции:
2025-09-27 22:17:26 deposit 922.0
2025-09-29 22:17:26 deposit 880.0
2025-10-12 22:17:26 deposit 838.0
Последние операции:
2026-10-04 22:03:28 deposit 25.0
2025-10-25 12:00:00 deposit 100.0
2025-10-25 12:00:00 deposit 100.0
